# 01 — Create datasets: **one engine (Spark), one format at a time**

**Purpose:** Generate the synthetic multimodal dataset and land it in **one** of four storage
formats, all on **pure Spark** — no Ray anywhere in the data-engineering path. This is the
*format-isolation* notebook for the blog: generation, write, and backfill are held constant on
Spark so the **only** variable is the terminal storage format.

Pick the format with the `format` widget:

| `format` | Image storage | Output |
|---|---|---|
| `delta_pathref` | JPEG **files** in a Volume + an `image_path` column | table `synthetic_delta_{size}` |
| `delta_inline_binary` | JPEG **bytes inline** in a `BINARY` column | table `synthetic_delta_inline_{size}` |
| `delta_inline_file` | JPEG **bytes inline** in a `FILE` column | table `synthetic_delta_inline_file_{size}` |
| `lance` | JPEG **bytes inline**, blob-isolated Lance fragments | dataset `synthetic_lance_{size}` |

The output names match exactly what `03_training_benchmark` reads and `04_compile_results`
compiles, so those notebooks are unchanged. Generation is the same deterministic `(SEED, id)`
logic for every format, wrapped in a `mapInArrow` UDF (the Arrow-batched Spark analog of a
vectorized map), so the bytes are byte-identical across the four routes.

Each run does **one** format; `02_run_benchmarks` submits the (format × size) grid as parallel
jobs, each on its own fresh cluster.

---

> ### ⚠️ Cluster requirements — read before running
>
> One Single-User **Unity-Catalog** cluster runs all four formats. The Delta paths need UC; the
> Lance path writes to a Volume too, but **not through the branded Spark connector** — see the box
> below for why, and what it uses instead.
>
> - **Runtime:** **DBR 16.4 LTS** (Spark 3.5 / Scala 2.12) or any DBR with a matching pyarrow/
>   pylance. Because the Lance path is pure pylance (no JVM connector), it is *not* pinned to the
>   `lance-spark-bundle`'s Spark 3.5 / Scala 2.12 build — one runtime serves all four formats.
> - **Access mode:** Single User (Dedicated). Unity Catalog enabled.
> - **Library (all formats):** `pip install pylance` (see the next cell). pylance drives the Lance
>   write, backfill, and verification directly — **no cluster JAR, no `ANY FILE` allowlist**.
>
> #### Why not the branded `lance-spark-bundle` connector?
>
> The "official" Spark route — install `org.lance:lance-spark-bundle-3.5_2.12`, register
> `spark.sql.catalog.lance = org.lance.spark.LanceNamespaceSparkCatalog`, and write via
> `df.write.format("lance")` / `ALTER TABLE … ADD COLUMNS … FROM` — **does not work on UC
> Single-User compute**: the DataSource connector and the catalog both fail there, the JAR needs
> UC allowlisting (`ANY FILE`), it's pinned to Spark 3.5 / Scala 2.12 (won't load on DBR 17.x), and
> its backfill SQL extension may not load on Databricks at all. Rather than fight all of that, this
> notebook takes the **same pylance-direct path as `01b_lance_native`**: each Spark task writes
> Lance fragments straight to the Volume's underlying `s3://` URI via `lance.fragment.write_fragments`
> (and `frag.merge_columns` for the backfill), with a single driver-side `LanceOperation` commit.
> So the "one variable is the storage format" claim holds — the engine is Spark throughout, but the
> Lance *connector* is deliberately not in the picture.
>
> #### The one commit-time constraint that IS fundamental
>
> Lance's default commit finalises a write with a POSIX `rename()`, which the UC Volume FUSE mount
> does not implement (OS error 38, `ENOSYS`). Writing to the Volume's `s3://` URI instead commits
> with an S3-native atomic `PutObject` — no rename needed. This is engine-agnostic (Ray, Spark, and
> plain Python all hit it) and is why every Lance write/commit below targets `lance_s3_path`, not
> the `/Volumes/...` FUSE path. The dataset still lands in the same Volume and reads back through
> `03`'s `read_lance`. Verify on your cluster that the Spark-written Lance dataset lands at
> `{volume}/synthetic_lance_{size}` with matching read throughput (fragment layout can differ from a
> Ray write).

In [0]:
# pylance is used only for driver-side Lance verification (fragment count, round-trip) and the
# backfill fallback. The Spark<->Lance write path itself comes from the lance-spark-bundle JAR
# installed as a CLUSTER LIBRARY (see the constraints cell) — NOT from pip.
%pip install -qU pylance numpy pandas "databricks-sdk>=0.49.0"
dbutils.library.restartPython()

In [0]:
# ── Widgets ─────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("seed", "42", "RNG seed")
dbutils.widgets.text("embedding_dim", "512", "Embedding dim")
# One format per run — the single variable this notebook isolates.
dbutils.widgets.dropdown("format", "lance", ["delta_pathref", "delta_inline_binary", "delta_inline_file", "lance"], "Storage format")
dbutils.widgets.text("lance_namespace", "lance", "Lance Spark catalog name (matches spark.sql.catalog.<name>)")

size          = dbutils.widgets.get("size")
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
volume        = dbutils.widgets.get("volume")
SEED          = int(dbutils.widgets.get("seed"))
EMBEDDING_DIM = int(dbutils.widgets.get("embedding_dim"))
FORMAT        = dbutils.widgets.get("format")
LANCE_CATALOG = dbutils.widgets.get("lance_namespace")

SIZE_MAP = {"10k": 10_000, "100k": 100_000, "1m": 1_000_000, "10m": 10_000_000}
N_ROWS   = SIZE_MAP[size]

# Fixed category set — MUST match 03_training_benchmark.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]

base_vol      = f"/Volumes/{catalog}/{schema}/{volume}"
images_dir    = f"{base_vol}/synthetic_images_{size}"                 # JPEG files (delta_pathref only)
delta_table   = f"{catalog}.{schema}.synthetic_delta_{size}"          # path-ref metadata table
inline_table  = f"{catalog}.{schema}.synthetic_delta_inline_{size}"   # inline-bytes table (BINARY)
inline_file_table = f"{catalog}.{schema}.synthetic_delta_inline_file_{size}"  # inline-bytes table (FILE type)
lance_subdir  = f"synthetic_lance_{size}"                             # 03 reads read_lance(base_vol/lance_subdir)
lance_path    = f"{base_vol}/{lance_subdir}"                          # physical Lance dataset (see constraints)
lance_full    = f"{LANCE_CATALOG}.default.{lance_subdir}"             # 3-level name via the dir namespace
artifacts_dir = f"{base_vol}/artifacts"

# Artifact filename per format — kept identical to what 04_compile_results loads.
ARTIFACT_NAME = {"delta_pathref":      f"delta_{size}.json",
                 "delta_inline_binary": f"delta_inline_{size}.json",
                 "delta_inline_file":   f"delta_inline_file_{size}.json",
                 "lance":              f"lance_{size}.json"}[FORMAT]
PATH_LABEL    = {"delta_pathref":      "delta_pathref",
                 "delta_inline_binary": "delta_inline_binary",
                 "delta_inline_file":   "delta_inline_file",
                 "lance":              "lance_native"}[FORMAT]

print(f"Size tier   : {size} ({N_ROWS:,} rows)")
print(f"Format      : {FORMAT}")
print(f"Artifact    : {artifacts_dir}/{ARTIFACT_NAME}")
if FORMAT == "delta_pathref":
    print(f"Output      : table {delta_table}  +  JPEG files under {images_dir}")
elif FORMAT == "delta_inline_binary":
    print(f"Output      : table {inline_table}  (BINARY column)")
elif FORMAT == "delta_inline_file":
    print(f"Output      : table {inline_file_table}  (FILE column)")
else:
    print(f"Output      : Lance dataset {lance_path}  (via {lance_full})")

In [0]:
# Lance writes use pylance + UC credential vending to write directly to the Volume's
# underlying S3 path (same architecture as 01b_lance_native). Each Spark task writes
# fragments in parallel; only metadata returns to the driver for the final commit.
# No JVM JAR, no Spark catalog, no FUSE rename — fully distributed, S3-native.
if FORMAT == "lance":
    import lance
    print(f"pylance version: {lance.__version__}")
    print(f"Lance FUSE path: {lance_path}")
    print(f"  (actual writes go to the Volume's S3 URI via credential vending)")
else:
    print(f"[{FORMAT}] Lance check skipped.")

In [0]:
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog as sdk_catalog

w = WorkspaceClient()
try:
    w.volumes.read(f"{catalog}.{schema}.{volume}")
except Exception:
    w.volumes.create(catalog_name=catalog, schema_name=schema, name=volume,
                     volume_type=sdk_catalog.VolumeType.MANAGED)
    print(f"Created volume {catalog}.{schema}.{volume}")
os.makedirs(artifacts_dir, exist_ok=True)
if FORMAT == "delta_pathref":
    os.makedirs(images_dir, exist_ok=True)


def dir_file_sizes(path):
    """path -> {file: size}. Lets us measure exactly the files a step ADDS (backfill),
    instead of differencing whole-directory totals (which can go negative when Lance
    version cleanup drops superseded files)."""
    sizes = {}
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try:
                sizes[fp] = os.path.getsize(fp)
            except OSError:
                pass
    return sizes


def dir_stats(path):
    s = dir_file_sizes(path)
    return sum(s.values()), len(s)

## Generate synthetic data (once, on Spark)

Identical `(SEED, id)` generation to the rest of the benchmark — reproduced verbatim so the bytes
match — but driven entirely by Spark. `spark.range(N)` produces the id column; `mapInArrow` fans
generation across executors as an **Arrow-batched** Python UDF (vectorized, per-batch Python — the
fair analog of a `map_batches`). The image is conditioned on category (hue) so `03`'s classifier is
learnable; noise keeps the JPEG ~30–300KB.

Generation is materialized behind a `cache()` + `count()` barrier so the **write timing below
excludes generation** — the write timer measures only the format-specific commit.

In [0]:
import numpy as np

# ── Verbatim generation logic so bytes are byte-identical across all formats (same seed -> same JPEG). ──
def _make_image(rng, category_idx, n_categories):
    """Procedural RGB image conditioned on category, JPEG-encoded to ~30-300KB."""
    import io
    from PIL import Image

    side = int(rng.integers(256, 512))
    base = np.zeros((side, side, 3), dtype=np.float32)
    hue = category_idx / n_categories
    base[..., 0] = 255 * hue
    base[..., 1] = 255 * (1 - hue)
    base[..., 2] = 128
    noise = rng.integers(0, 60, size=(side, side, 3))
    arr = np.clip(base + noise, 0, 255).astype(np.uint8)

    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="JPEG", quality=90)
    return buf.getvalue()


def _generate_rows(ids, seed, categories, embedding_dim):
    """Per-batch generation returning plain Python lists for Arrow assembly."""
    n_cat = len(categories)
    images, captions, embeddings, cats, brightness, quality = [], [], [], [], [], []
    for _id in ids:
        rng = np.random.default_rng([seed, int(_id)])
        cat_idx = int(rng.integers(0, n_cat))
        images.append(_make_image(rng, cat_idx, n_cat))
        captions.append(f"a photo of a {categories[cat_idx]} " + "x" * int(rng.integers(0, 40)))
        embeddings.append(rng.standard_normal(embedding_dim).astype(np.float32).tolist())
        cats.append(categories[cat_idx])
        brightness.append(float(rng.random()))
        quality.append(int(rng.integers(1, 6)))
    return images, captions, embeddings, cats, brightness, quality

In [0]:
import pyarrow as pa
from pyspark.sql.types import (
    StructType, StructField, LongType, BinaryType, StringType,
    ArrayType, FloatType, IntegerType,
)

# Generated DataFrame schema. Arrow<->Spark: BinaryType<->binary, ArrayType(FloatType)<->list<float32>.
GEN_SCHEMA = StructType([
    StructField("id",         LongType(),   False),
    StructField("image",      BinaryType(), False),
    StructField("caption",    StringType(), False),
    StructField("embedding",  ArrayType(FloatType()), False),
    StructField("category",   StringType(), False),
    StructField("brightness", FloatType(),  False),
    StructField("quality",    IntegerType(), False),
])

_SEED, _CATS, _DIM = SEED, CATEGORIES, EMBEDDING_DIM

def generate_arrow(batch_iter):
    """mapInArrow UDF: iterator of pa.RecordBatch (with an `id` column) -> generated batches."""
    for rb in batch_iter:
        ids = rb.column("id").to_pylist()
        images, captions, embeddings, cats, brightness, quality = _generate_rows(
            ids, _SEED, _CATS, _DIM)
        yield pa.record_batch({
            "id":         pa.array(ids, type=pa.int64()),
            "image":      pa.array(images, type=pa.binary()),
            "caption":    pa.array(captions, type=pa.string()),
            "embedding":  pa.array(embeddings, type=pa.list_(pa.float32())),
            "category":   pa.array(cats, type=pa.string()),
            "brightness": pa.array(brightness, type=pa.float32()),
            "quality":    pa.array(quality, type=pa.int32()),
        })

# Partition count controls generation parallelism and downstream fragment/file count
# (~5k rows/partition, floor 64) — a comparable layout across formats.
n_parts = max(64, N_ROWS // 5_000)
ids_df  = spark.range(0, N_ROWS, numPartitions=n_parts)          # column: id
gen_df  = ids_df.mapInArrow(generate_arrow, schema=GEN_SCHEMA)

# Materialize generation BEHIND A BARRIER so the write timing excludes generation.
gen_df = gen_df.cache()
row_count = gen_df.count()
raw_image_bytes = gen_df.selectExpr("sum(length(image)) AS b").collect()[0]["b"]
print(f"Generated {row_count:,} rows (cached) | raw image bytes: {raw_image_bytes / 1e9:.3f} GB")

## Write — the one format this run isolates

All four routes start from the **same cached `gen_df`** and diverge only here:

- **`delta_pathref`** — a `mapInArrow` writes each JPEG to the Volume and returns the row with an
  `image_path` (no bytes); that metadata DataFrame is written to a Delta table. Cost = the file
  PUT storm + the small metadata table.
- **`delta_inline_binary`** — `gen_df` written straight to a Delta table with the JPEG **bytes inline** in
  a `BINARY` column. Every image byte funnels through the Spark write path.
- **`delta_inline_file`** — same as above but the image column uses the new `FILE` type
  ([docs](https://docs.databricks.com/aws/en/unstructured/file)), which stores bytes inline with
  file-aware semantics (content-type inference, streaming reads, etc.).
- **`lance`** — a `mapInArrow` UDF has each Spark task write its rows straight to Lance fragments on
  the Volume's `s3://` URI (`lance.fragment.write_fragments`), returning only serialized fragment
  metadata; a single driver-side `LanceOperation.Overwrite` commit stitches them into one dataset
  version. The JPEG bytes land **inline** in the fragments — the apples-to-apples counterpart to the
  inline Delta bytes — and the whole write is distributed, S3-native, and never touches the Spark
  Lance connector or the FUSE `/Volumes/...` path.

In [0]:
import time

n_output_files = None
on_disk_bytes  = None

if FORMAT == "delta_pathref":
    # Write JPEG files via an Arrow UDF, returning metadata rows with image_path (no bytes).
    _images_dir = images_dir
    META_SCHEMA = StructType([
        StructField("id",         LongType(),   False),
        StructField("image_path", StringType(), False),
        StructField("caption",    StringType(), False),
        StructField("embedding",  ArrayType(FloatType()), False),
        StructField("category",   StringType(), False),
        StructField("brightness", FloatType(),  False),
        StructField("quality",    IntegerType(), False),
    ])

    def write_files_arrow(batch_iter):
        import os
        for rb in batch_iter:
            ids   = rb.column("id").to_pylist()
            imgs  = rb.column("image").to_pylist()
            paths = []
            for _id, jpeg in zip(ids, imgs):
                p = os.path.join(_images_dir, f"{int(_id):012d}.jpg")
                with open(p, "wb") as f:
                    f.write(jpeg)
                paths.append(p)
            yield pa.record_batch({
                "id":         pa.array(ids, type=pa.int64()),
                "image_path": pa.array(paths, type=pa.string()),
                "caption":    rb.column("caption"),
                "embedding":  rb.column("embedding"),
                "category":   rb.column("category"),
                "brightness": rb.column("brightness"),
                "quality":    rb.column("quality"),
            })

    spark.sql(f"DROP TABLE IF EXISTS {delta_table}")
    t0 = time.time()
    meta_df = gen_df.mapInArrow(write_files_arrow, schema=META_SCHEMA)
    # One action: writes the JPEG files AND the metadata table (files land as a side effect).
    meta_df.write.mode("overwrite").saveAsTable(delta_table)
    write_s = time.time() - t0

    img_bytes, img_files = dir_stats(images_dir)
    meta_bytes = spark.sql(f"DESCRIBE DETAIL {delta_table}").collect()[0]["sizeInBytes"] or 0
    on_disk_bytes  = int(img_bytes + meta_bytes)
    n_output_files = int(img_files + 1)                       # JPEGs + the metadata Parquet
    tgt_table      = delta_table
    print(f"delta_pathref : {write_s:6.2f}s | {img_files:,} JPEG files + 1 metadata table | "
          f"{on_disk_bytes / 1e9:.3f} GB")

elif FORMAT == "delta_inline_binary":
    spark.sql(f"DROP TABLE IF EXISTS {inline_table}")
    t0 = time.time()
    gen_df.write.mode("overwrite").saveAsTable(inline_table)  # JPEG bytes inline in a BINARY column
    write_s = time.time() - t0

    _d = spark.sql(f"DESCRIBE DETAIL {inline_table}").collect()[0]
    on_disk_bytes  = int(_d["sizeInBytes"] or 0)
    n_output_files = int(_d["numFiles"] or 0)                 # Parquet files only (bytes are inline)
    tgt_table      = inline_table
    print(f"delta_inline_binary: {write_s:6.2f}s | BINARY column across {n_output_files} Parquet files | "
          f"{on_disk_bytes / 1e9:.3f} GB")

elif FORMAT == "delta_inline_file":
    # FILE type: new unstructured-data column type that stores bytes inline with file semantics.
    # https://docs.databricks.com/aws/en/unstructured/file
    spark.sql(f"DROP TABLE IF EXISTS {inline_file_table}")
    # Create table with FILE type column, then append data (Spark writes binary -> FILE transparently).
    spark.sql(f"""
        CREATE TABLE {inline_file_table} (
            id         BIGINT       NOT NULL,
            image      FILE,
            caption    STRING       NOT NULL,
            embedding  ARRAY<FLOAT> NOT NULL,
            category   STRING       NOT NULL,
            brightness FLOAT        NOT NULL,
            quality    INT          NOT NULL
        )
    """)
    t0 = time.time()
    gen_df.writeTo(inline_file_table).append()
    write_s = time.time() - t0

    _d = spark.sql(f"DESCRIBE DETAIL {inline_file_table}").collect()[0]
    on_disk_bytes  = int(_d["sizeInBytes"] or 0)
    n_output_files = int(_d["numFiles"] or 0)
    tgt_table      = inline_file_table
    print(f"delta_inline_file : {write_s:6.2f}s | FILE column across {n_output_files} Parquet files | "
          f"{on_disk_bytes / 1e9:.3f} GB")

else:  # lance
    # Write via pylance directly — bypasses both the Spark catalog AND the Spark DataSource
    # connector (both fail on UC SINGLE_USER clusters).
    # FUSE limitation: /Volumes/ doesn't support rename() (OS error 38) which Lance's commit
    # protocol requires. So: write to LOCAL disk first, then copy to the Volume.
    import shutil, lance

    # ── Distributed Lance write directly to S3 (same pattern as 01b_lance_native) ──
    # Each Spark task writes Lance fragments directly to the Volume's underlying S3 path
    # using UC credential vending. Only tiny fragment metadata returns to the driver for
    # the final atomic commit. NO data collection, NO FUSE rename, fully distributed.
    import base64, pickle, requests
    import lance
    from lance.fragment import write_fragments
    import boto3

    # Resolve the S3 URI backing the managed Volume
    vol_info = w.volumes.read(f"{catalog}.{schema}.{volume}")
    lance_s3_path = f"{vol_info.storage_location.rstrip('/')}/{lance_subdir}"
    _vol_id = vol_info.volume_id
    _aws_region = boto3.session.Session().region_name or "us-west-2"
    _db_host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    _db_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

    print(f"  S3 target: {lance_s3_path}")
    print(f"  FUSE mirror: {lance_path}")

    def _s3_storage_options() -> dict:
        """UC credential vending — temporary S3 credentials for the managed volume."""
        resp = requests.post(
            f"{_db_host}/api/2.1/unity-catalog/temporary-volume-credentials",
            headers={"Authorization": f"Bearer {_db_token}"},
            json={"volume_id": _vol_id, "operation": "WRITE_VOLUME"},
            timeout=10,
        )
        if not resp.ok:
            raise RuntimeError(f"UC credential vending {resp.status_code}: {resp.text}")
        aws = resp.json()["aws_temp_credentials"]
        return {
            "aws_access_key_id":     aws["access_key_id"],
            "aws_secret_access_key": aws["secret_access_key"],
            "aws_session_token":     aws.get("session_token", ""),
            "aws_region":            _aws_region,
        }

    # Broadcast credentials + paths to all workers
    _bc_s3_path = spark.sparkContext.broadcast(lance_s3_path)
    _bc_host = spark.sparkContext.broadcast(_db_host)
    _bc_token = spark.sparkContext.broadcast(_db_token)
    _bc_vol_id = spark.sparkContext.broadcast(_vol_id)
    _bc_region = spark.sparkContext.broadcast(_aws_region)

    def _write_frags_partition(iterator):
        """UDF: each Spark task writes Lance fragments to S3, returns serialized metadata."""
        import base64, pickle, requests
        import pyarrow as pa
        import numpy as np
        from lance.fragment import write_fragments

        s3_path = _bc_s3_path.value
        host, token = _bc_host.value, _bc_token.value
        vol_id, region = _bc_vol_id.value, _bc_region.value

        def get_creds():
            resp = requests.post(
                f"{host}/api/2.1/unity-catalog/temporary-volume-credentials",
                headers={"Authorization": f"Bearer {token}"},
                json={"volume_id": vol_id, "operation": "WRITE_VOLUME"},
                timeout=10,
            )
            aws = resp.json()["aws_temp_credentials"]
            return {
                "aws_access_key_id": aws["access_key_id"],
                "aws_secret_access_key": aws["secret_access_key"],
                "aws_session_token": aws.get("session_token", ""),
                "aws_region": region,
            }

        for batch in iterator:
            fragments = write_fragments(batch, s3_path, schema=batch.schema, storage_options=get_creds())
            for frag in fragments:
                yield pa.RecordBatch.from_pydict({
                    "fragment_b64": [base64.b64encode(pickle.dumps(frag)).decode("ascii")],
                    "schema_b64": [base64.b64encode(pickle.dumps(batch.schema)).decode("ascii")],
                })

    t0 = time.time()
    # Distributed fragment writes across all 8 workers
    fragment_df = gen_df.mapInArrow(_write_frags_partition, schema="fragment_b64 STRING, schema_b64 STRING")
    fragment_rows = fragment_df.collect()

    # Driver-side commit — merge fragment metadata into a single dataset version
    fragments, lance_schema = [], None
    for row in fragment_rows:
        fragments.append(pickle.loads(base64.b64decode(row["fragment_b64"])))
        lance_schema = pickle.loads(base64.b64decode(row["schema_b64"]))

    op = lance.LanceOperation.Overwrite(lance_schema, fragments)
    lance.LanceDataset.commit(lance_s3_path, op, storage_options=_s3_storage_options())
    write_s = time.time() - t0
    print(f"  Distributed write + commit: {write_s:.1f}s")

    # Fragment count + on-disk bytes via pylance.
    n_frag = None
    try:
        lds = lance.dataset(lance_s3_path, storage_options=_s3_storage_options())
        n_frag = len(lds.get_fragments())
        on_disk_bytes, _ = dir_stats(lance_path)  # FUSE mirror is readable after commit
    except Exception as e:
        print(f"[warn] pylance open at {lance_s3_path} failed ({e})")
        on_disk_bytes, _ = dir_stats(lance_path)
    n_output_files = int(n_frag) if n_frag is not None else None
    tgt_table      = lance_path  # no catalog table — just a path
    print(f"lance         : {write_s:6.2f}s | "
          f"{n_frag if n_frag is not None else '?'} fragments | "
          f"{(on_disk_bytes or 0) / 1e9:.3f} GB "
          f"(vs ~{N_ROWS:,} JPEG PUTs on the path-ref route)")

## Verify — deterministic round-trip

Regenerate the probe rows from the same `(SEED, id)` and confirm the stored bytes match
byte-for-byte — proving the Spark write is lossless and identical across formats. The read stays
pure Spark (`spark.read.table`) for Delta; Lance reads back through the catalog too.

In [0]:
probe_ids = [0, N_ROWS // 2, N_ROWS - 1]

def _fetch_stored(pids):
    """Return {id: image_bytes} for probe ids, per format."""
    if FORMAT == "delta_pathref":
        rows = (spark.read.table(delta_table)
                .where(f"id IN ({','.join(map(str, pids))})")
                .select("id", "image_path").collect())
        out = {}
        for r in rows:
            with open(r["image_path"], "rb") as f:
                out[r["id"]] = f.read()
        return out
    if FORMAT == "delta_inline_binary":
        rows = (spark.read.table(inline_table)
                .where(f"id IN ({','.join(map(str, pids))})")
                .select("id", "image").collect())
        return {r["id"]: bytes(r["image"]) for r in rows}
    if FORMAT == "delta_inline_file":
        rows = (spark.read.table(inline_file_table)
                .where(f"id IN ({','.join(map(str, pids))})")
                .select("id", "image").collect())
        # FILE column returns bytes (or a struct with content); extract raw bytes.
        def _extract_bytes(val):
            if isinstance(val, (bytes, bytearray)):
                return bytes(val)
            # FILE type may return a struct with a 'bytes' or 'content' field
            if hasattr(val, "content"):
                return bytes(val.content)
            if hasattr(val, "bytes"):
                return bytes(val.bytes)
            return bytes(val)
        return {r["id"]: _extract_bytes(r["image"]) for r in rows}
    else:  # lance — read from S3 via pylance (same path we wrote to)
        import lance
        ds = lance.dataset(lance_s3_path, storage_options=_s3_storage_options())
        tbl = ds.to_table(filter=f"id IN ({','.join(map(str, pids))})", columns=["id", "image"])
        return {int(row["id"]): bytes(row["image"]) for row in tbl.to_pylist()}

def _regen_image(pid):
    rng = np.random.default_rng([SEED, int(pid)])
    cat_idx = int(rng.integers(0, len(CATEGORIES)))
    return _make_image(rng, cat_idx, len(CATEGORIES))

stored = _fetch_stored(probe_ids)
roundtrip_ok = True
print("Round-trip (regenerated bytes == stored):")
for pid in probe_ids:
    ok = _regen_image(pid) == stored.get(pid)
    roundtrip_ok = roundtrip_ok and ok
    kb = len(stored.get(pid, b"")) / 1024
    print(f"  id={pid:>12,}: {'OK' if ok else 'MISMATCH':>8}  ({kb:.0f} KB)")
print(f"\nround-trip all OK: {roundtrip_ok}")

## ETL backfill — add a derived column (`embedding_norm`)

Add the L2 norm of the embedding — a stand-in for any derived feature — to the written dataset.
This is where the format difference is structural, all on the **same Spark engine**:

- **Delta (all three)** — `ALTER TABLE ADD COLUMN` + `UPDATE`. The `UPDATE` rewrites whole Parquet row
  groups; for **inline** Delta (BINARY or FILE) that drags every image byte along (measured via the `UPDATE`
  operation's own `numAddedBytes`, the true rewrite cost — not the post-`UPDATE` table size,
  which double-counts the retained pre-`VACUUM` version).
- **Lance** — a **distributed `merge_columns` backfill** that mirrors the write path above, for the
  same reason the write does: the Lance Spark SQL extension (`ALTER TABLE … ADD COLUMNS … FROM`)
  doesn't load on UC Single-User compute, so the branded path isn't available. Instead a
  `mapInArrow` UDF fans one `frag.merge_columns()` per fragment across the Spark workers — each
  writes **only the new column file** for its fragment (data files untouched) — then a driver-side
  `LanceOperation.Merge` commits the updated fragment metadata to the Volume's `s3://` URI, exactly
  as the write's `Overwrite` commit does. Same four-step shape as `01b_lance_native`'s Ray backfill,
  just with Spark tasks instead of Ray tasks doing the fan-out.

In [0]:
backfill_via = None

if FORMAT in ("delta_pathref", "delta_inline_binary", "delta_inline_file"):
    tbl = delta_table if FORMAT == "delta_pathref" else (inline_file_table if FORMAT == "delta_inline_file" else inline_table)
    _existing = {f.name for f in spark.table(tbl).schema.fields}
    t0 = time.time()
    if "embedding_norm" not in _existing:
        spark.sql(f"ALTER TABLE {tbl} ADD COLUMN embedding_norm FLOAT")
    spark.sql(f"""
        UPDATE {tbl}
        SET embedding_norm = SQRT(AGGREGATE(TRANSFORM(embedding, x -> x * x),
                                            CAST(0.0 AS DOUBLE), (acc, v) -> acc + v))
    """)
    backfill_s = time.time() - t0

    # Real bytes/files rewritten — from the UPDATE's own operationMetrics, not table size.
    _hist = spark.sql(f"DESCRIBE HISTORY {tbl}").collect()
    _upd  = next(r for r in _hist if r["operation"] == "UPDATE")
    _m    = _upd["operationMetrics"] or {}
    etl_bytes_written = int(_m.get("numAddedBytes") or _m.get("rewrittenBytes") or 0)
    etl_files         = int(_m.get("numAddedFiles") or 0)
    backfill_via = "spark_update"
    print(f"{FORMAT} backfill: {backfill_s:6.2f}s | rewrote {etl_bytes_written / 1e6:,.1f} MB "
          f"across {etl_files} files")

else:  # lance — DISTRIBUTED backfill via Spark-parallel merge_columns
    # Same pattern as the Ray version (01b_lance_native cell 16): one task per fragment
    # calls frag.merge_columns() which writes ONLY the new column file for that fragment.
    # Driver-side LanceOperation.Merge commits the updated fragment metadata atomically.
    # This replaces the old single-threaded lds_rw.add_columns() that serialized 1500+
    # S3 PUTs on the driver — now they fan across all Spark workers in parallel.
    import lance, pickle, base64

    before = dir_file_sizes(lance_path)  # FUSE mirror

    lds_rw = lance.dataset(lance_s3_path, storage_options=_s3_storage_options())
    if "embedding_norm" in lds_rw.schema.names:
        lds_rw.drop_columns(["embedding_norm"])
        lds_rw = lance.dataset(lance_s3_path, storage_options=_s3_storage_options())

    read_version = lds_rw.version
    frag_ids = [f.fragment_id for f in lds_rw.get_fragments()]
    print(f"  Backfilling {len(frag_ids)} fragments in parallel across Spark workers...")

    # Broadcast what each worker needs
    _bc_frag_s3_path = spark.sparkContext.broadcast(lance_s3_path)
    _bc_frag_host = spark.sparkContext.broadcast(_db_host)
    _bc_frag_token = spark.sparkContext.broadcast(_db_token)
    _bc_frag_vol_id = spark.sparkContext.broadcast(_vol_id)
    _bc_frag_region = spark.sparkContext.broadcast(_aws_region)

    def _merge_norm_partition(iterator):
        """Spark mapInArrow UDF: each task merge_columns on its assigned fragments."""
        import pickle, base64, requests
        import numpy as np
        import pyarrow as pa
        import lance

        s3_path = _bc_frag_s3_path.value
        host, token = _bc_frag_host.value, _bc_frag_token.value
        vol_id, region = _bc_frag_vol_id.value, _bc_frag_region.value

        def get_creds():
            resp = requests.post(
                f"{host}/api/2.1/unity-catalog/temporary-volume-credentials",
                headers={"Authorization": f"Bearer {token}"},
                json={"volume_id": vol_id, "operation": "WRITE_VOLUME"},
                timeout=10,
            )
            aws = resp.json()["aws_temp_credentials"]
            return {
                "aws_access_key_id": aws["access_key_id"],
                "aws_secret_access_key": aws["secret_access_key"],
                "aws_session_token": aws.get("session_token", ""),
                "aws_region": region,
            }

        def compute_norm(record_batch):
            embs = np.stack(record_batch.column("embedding").to_pylist()).astype("float32")
            norms = np.linalg.norm(embs, axis=1).astype("float32")
            return pa.record_batch({"embedding_norm": pa.array(norms)})

        for batch in iterator:
            fids = batch.column("frag_id").to_pylist()
            for fid in fids:
                ds = lance.dataset(s3_path, storage_options=get_creds())
                frag = ds.get_fragment(fid)
                new_frag_meta, new_schema = frag.merge_columns(
                    compute_norm, columns=["embedding"]
                )
                yield pa.record_batch({
                    "frag_meta_b64": [base64.b64encode(pickle.dumps(new_frag_meta)).decode("ascii")],
                    "schema_b64": [base64.b64encode(pickle.dumps(new_schema)).decode("ascii")],
                })

    # Distribute fragment IDs across workers — one partition per ~4 fragments for parallelism
    n_backfill_parts = max(8, len(frag_ids) // 4)
    frag_df = spark.createDataFrame([(fid,) for fid in frag_ids], ["frag_id"])
    frag_df = frag_df.repartition(n_backfill_parts)

    t0 = time.time()
    result_df = frag_df.mapInArrow(
        _merge_norm_partition, schema="frag_meta_b64 STRING, schema_b64 STRING"
    )
    result_rows = result_df.collect()

    # Driver-side commit — stitch updated fragment metadata into a new dataset version
    new_fragments = [pickle.loads(base64.b64decode(r["frag_meta_b64"])) for r in result_rows]
    new_schema = pickle.loads(base64.b64decode(result_rows[0]["schema_b64"]))

    op = lance.LanceOperation.Merge(new_fragments, new_schema)
    lance.LanceDataset.commit(
        lance_s3_path, op, read_version=read_version,
        storage_options=_s3_storage_options()
    )
    backfill_via = "spark_parallel_merge_columns"
    backfill_s = time.time() - t0

    after = dir_file_sizes(lance_path)
    etl_bytes_written = sum(sz for fp, sz in after.items() if fp not in before)
    etl_files = len([fp for fp in after if fp not in before])
    print(f"lance backfill : {backfill_s:6.2f}s | +{etl_bytes_written / 1e6:,.1f} MB new-column bytes "
          f"| {len(frag_ids)} fragments merged in parallel | via {backfill_via}")

In [0]:
import json

# ── common block ── identical key names across all formats so 04_compile_results stacks them
# into one table with no per-format mapping. Format-specific detail stays in `raw`.
# Generation ran behind the cache() barrier, so write_total_s == target_write_s (no separate
# file-landing step timed apart from the write).
common = {
    "path_label":        PATH_LABEL,
    "write_total_s":     round(write_s, 3),
    "target_write_s":    round(write_s, 3),
    "n_output_files":    n_output_files,
    "on_disk_bytes":     int(on_disk_bytes) if on_disk_bytes is not None else None,
    "etl_backfill_s":    round(backfill_s, 3),
    "etl_bytes_written": int(etl_bytes_written),
    "roundtrip_ok":      bool(roundtrip_ok),
}

metrics = {
    "size":   size,
    "n_rows": int(N_ROWS),
    "common": common,
    "raw": {
        "engine":            "spark",
        "format":            FORMAT,
        "raw_image_gb":      round(raw_image_bytes / 1e9, 4),
        "write_s":           round(write_s, 3),
        "on_disk_bytes":     int(on_disk_bytes) if on_disk_bytes is not None else None,
        "n_output_files":    n_output_files,
        "backfill_s":        round(backfill_s, 3),
        "backfill_via":      backfill_via,
        "etl_bytes_written": int(etl_bytes_written),
        "etl_files":         int(etl_files),
        "n_partitions":      int(n_parts),
        "roundtrip_ok":      bool(roundtrip_ok),
    },
}
out_path = f"{artifacts_dir}/{ARTIFACT_NAME}"
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Wrote {out_path}")
print(json.dumps(metrics, indent=2))

## Done — dataset + artifact ready (pure Spark)

The `{FORMAT}` dataset was generated, written, verified, and backfilled **entirely on Spark**. Its
metrics are in `artifacts/{ARTIFACT_NAME}` with the same `common` keys the other formats use, so
`04_compile_results` stacks all four formats directly.

**Next:** run this notebook for the other formats/sizes (or let `02_run_benchmarks` submit the
grid), then `03_training_benchmark` (Ray Train DDP — the natural home for distributed training)
reads each dataset back, and `04_compile_results` compiles the head-to-head.